# Data loading and inscpecting

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("fake_reviews_dataset.csv")

# Shape, column and data types of data
print(f"Shape (rows, columns): {df.shape}")
print(f"Columns of dataset: {df.columns.tolist()}")
print(f"Data types of dataset: {df.dtypes}")

df.head(2)

Shape (rows, columns): (40432, 4)
Columns of dataset: ['category', 'rating', 'label', 'text_']
Data types of dataset: category     object
rating      float64
label        object
text_        object
dtype: object


,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40432 entries, 0 to 40431
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   category  40432 non-null  object 
 1   rating    40432 non-null  float64
 2   label     40432 non-null  object 
 3   text_     40432 non-null  object 
dtypes: float64(1), object(3)
memory usage: 1.2+ MB


In [4]:
df.describe()

,rating
count,40432.000000
mean,4.256579
std,1.144354
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


In [6]:
## Checking for missing values

missing = df.isnull().sum()
missing_pct = (missing/len(df))*100

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_pct" : missing_pct.round(2)
}).sort_values("missing_count", ascending= False)

print(missing_summary[missing_summary["missing_count"]>0])

Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []


In [8]:
# Duplicate rows
print(f"Full duplicate rows: {df.duplicated().sum()}")

Full duplicate rows: 12


In [14]:
# Label Column - Class Balance
print(f"Unique lables: {df["label"].unique()}")
print(f"Label counts: {df["label"].value_counts()}")
print(f"Label proportions (%):\n {df["label"].value_counts(normalize=True)*100}")

Unique lables: ['CG' 'OR']
Label counts: label
CG    20216
OR    20216
Name: count, dtype: int64
Label proportions (%):
 label
CG    50.0
OR    50.0
Name: proportion, dtype: float64


In [16]:
# Rating Distribution
print(f"Unique rating: {sorted(df["label"].unique())}")
print(f"Rating counts: \n{df["rating"].value_counts().sort_index()}")

Unique rating: ['CG', 'OR']
Rating counts: 
rating
1.0     2155
2.0     1967
3.0     3786
4.0     7965
5.0    24559
Name: count, dtype: int64


In [17]:
# Category distribution
print(f"Number of categories: {df["category"].nunique()}")
print(f"Category counts: \n{df["category"].value_counts()}")

Number of categories: 10
Category counts: 
category
Kindle_Store_5                  4730
Books_5                         4370
Pet_Supplies_5                  4254
Home_and_Kitchen_5              4056
Electronics_5                   3988
Sports_and_Outdoors_5           3946
Tools_and_Home_Improvement_5    3858
Clothing_Shoes_and_Jewelry_5    3848
Toys_and_Games_5                3794
Movies_and_TV_5                 3588
Name: count, dtype: int64


In [18]:
# Quick look at review text length, before any cleaning
df["text_"].str.len().describe()

count    40432.000000
mean       351.271963
std        369.813570
min         24.000000
25%        107.000000
50%        198.000000
75%        439.000000
max       2827.000000
Name: text_, dtype: float64

#Data cleaning

In [19]:
# work on a copy
df_clean = df.copy()

# drop duplicate rows
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Dropped {before -len(df_clean)} duplicate rows")

Dropped 12 duplicate rows


In [20]:
# Encode label as binary:1 fake(CG), 0 genuine (OR)
df_clean["is_fake"] = df_clean["label"].map({"CG":1, "OR": 0})

# Sanity check - must be 0
print("Unmapped rows: ", df_clean["is_fake"].isnull().sum())
print("\n Class balance: \n", df_clean["is_fake"].value_counts(normalize=True)*100)

Unmapped rows:  0

 Class balance: 
 is_fake
0    50.01237
1    49.98763
Name: proportion, dtype: float64


In [21]:
# 2.4 Clean review text — strip whitespace, drop empty reviews
df_clean["text_"] = df_clean["text_"].astype(str).str.strip()

before = len(df_clean)
df_clean = df_clean[df_clean["text_"].str.len() > 0]
print(f"Dropped {before - len(df_clean)} rows with empty review text")

Dropped 0 rows with empty review text


In [22]:
# 2.5 Clean category column — strip the "_5" suffix for readability
# (e.g. "Home_and_Kitchen_5" -> "Home and Kitchen")
df_clean["category_clean"] = (
    df_clean["category"]
    .str.replace("_5", "", regex=False)
    .str.replace("_", " ", regex=False)
)

df_clean["category_clean"].value_counts()

category_clean
Kindle Store                  4728
Books                         4369
Pet Supplies                  4251
Home and Kitchen              4056
Electronics                   3988
Sports and Outdoors           3944
Tools and Home Improvement    3858
Clothing Shoes and Jewelry    3847
Toys and Games                3792
Movies and TV                 3587
Name: count, dtype: int64

In [23]:
# 2.6 Create derived features (matches your paper's feature set, minus behavioral cols)
df_clean["review_length"] = df_clean["text_"].str.len()
df_clean["word_count"] = df_clean["text_"].str.split().str.len()

df_clean[["review_length", "word_count"]].describe()

,review_length,word_count
count,40420.000000,40420.000000
mean,351.298144,67.474592
std,369.852346,69.588164
min,5.000000,1.000000
25%,107.000000,21.000000
50%,198.000000,39.000000
75%,439.000000,85.000000
max,2827.000000,373.000000


In [24]:
# 2.7 Final check before EDA
print("Final shape:", df_clean.shape)
df_clean.head()

# Save cleaned version
df_clean.to_csv("fake_reviews_cleaned.csv", index=False)
print("\nSaved cleaned dataset.")

Final shape: (40420, 8)

Saved cleaned dataset.
